# SmartDs 2D Slice Plot (Starter)

This notebook reads an already-2D BATSRUS slice file and plots a **flat-shaded triangulated field** on the native 2D mesh.

It reuses the library (`SmartDs`) and the file's native mesh connectivity (`sds.corners`) and does **not** resample.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

from starwinds_analysis.smart_ds import SmartDs
from starwinds_analysis.data.samples import get_sample
from starwinds_analysis.utils import auto_coords, triangles

plt.rcParams['figure.dpi'] = 120


## Choose a 2D File

By default this looks for a `z=0*.plt` slice in `sample_data/`.

In [ ]:
# Minimal sample lookup for examples/tests (reused helper)
DATA_FILE = Path(get_sample('z=0_var_3_n00060000.plt'))
print('Using:', DATA_FILE)

# TODO like i wrote, THIS should be just "print(sds)" and the __str__ method should be implemented to show all this info in a nice format. 
sds = SmartDs.from_file(DATA_FILE)
print('Title:', sds.title)
print('Zone :', sds.zone)
print('Points:', len(sds.points))
print('Variables:', len(sds.variables))


In [ ]:
# Inspect available fields
list(sds.variables)[:40]

## Auto-Detect Slice Coordinates and Choose a Field

The notebook inspects `X [R]`, `Y [R]`, `Z [R]` and picks the two coordinates that vary across the 2D slice.

In [ ]:
X_FIELD, Y_FIELD = auto_coords(sds)
COLOR_FIELD = 'Rho [g/cm^3]'  # change as needed
tris = triangles(sds, X_FIELD, Y_FIELD)

print('Detected in-plane coordinates:', X_FIELD, Y_FIELD)
print('Triangles:', tris.triangles.shape[0])


In [ ]:
# Triangulation built from the native 2D mesh connectivity (no resampling)
tris


In [ ]:
# Flat-shaded triangulated plot on the native 2D mesh (no resampling)
COLOR_SCALE = 'auto'  # 'auto', 'linear', or 'log'

c = np.asarray(sds.variable(COLOR_FIELD), dtype=float)
c_plot = np.ma.masked_invalid(c)

norm = None
if COLOR_SCALE == 'log' or (COLOR_SCALE == 'auto' and np.all(c_plot.compressed() > 0)):
    c_plot = np.ma.masked_less_equal(c_plot, 0.0)
    norm = LogNorm()

fig, ax = plt.subplots(figsize=(7, 6))
img = ax.tripcolor(tris, c_plot, shading='flat', cmap='viridis', norm=norm)
ax.triplot(tris, color='k', linewidth=0.15, alpha=0.15)
ax.set_aspect('equal')
ax.set_xlabel(X_FIELD)
ax.set_ylabel(Y_FIELD)
ax.set_title(COLOR_FIELD)
fig.colorbar(img, ax=ax, label=COLOR_FIELD)
plt.show()


## Alfvén Surface (2D Contour)

This uses the BATSRUS recipe graph to compute the Alfvén Mach number `M_A` on demand and draws the `M_A = 1` contour as a 2D Alfvén-surface proxy in the slice plane.

In [ ]:
# 2D Alfvén-surface proxy: M_A = 1 contour on the native slice mesh
sds.add_batsrus_graph()
ma = np.asarray(sds.variable('M_A [none]'), dtype=float)
ma_plot = np.ma.masked_less_equal(np.ma.masked_invalid(ma), 0.0)

cmap = plt.get_cmap('cividis').copy()
cmap.set_under(cmap(0.0))
cmap.set_over(cmap(1.0))

fig, ax = plt.subplots(figsize=(7, 6))
img = ax.tripcolor(
    tris,
    ma_plot,
    shading='flat',
    cmap=cmap,
    norm=LogNorm(vmin=1e-2, vmax=1e2),
)
ax.tricontour(tris, ma, levels=[1.0], colors='crimson', linewidths=1.5)
ax.set_aspect('equal')
ax.set_xlabel(X_FIELD)
ax.set_ylabel(Y_FIELD)
ax.set_title('Alfvén Mach number')
fig.colorbar(img, ax=ax, label='M_A', extend='both')
plt.show()
